In [1]:
# Cell 1: Load Data and Investigate Anomalies
import pandas as pd
import numpy as np

# 1. Load the clean, imputed dataset from the previous step
df_clean = pd.read_csv('application_train_imputed.csv')
print(f"Data Loaded: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns")

# 2. Investigate Numerical Anomalies using Summary Statistics
print("\n--- Numerical Anomaly Check (Min & Max) ---")
numeric_cols = df_clean.select_dtypes(include=['number']).columns

# Create a summary dataframe showing Min, Max, and Median
anomaly_check = pd.DataFrame({
    'Min': df_clean[numeric_cols].min(),
    'Median': df_clean[numeric_cols].median(),
    'Max': df_clean[numeric_cols].max()
})

# Display the summary
print(anomaly_check)

Data Loaded: 307511 rows, 40 columns

--- Numerical Anomaly Check (Min & Max) ---
                                 Min    Median          Max
CNT_CHILDREN                     0.0       0.0         19.0
AMT_INCOME_TOTAL             25650.0  147150.0  117000000.0
AMT_CREDIT                   45000.0  513531.0    4050000.0
AMT_ANNUITY                   1615.5   24903.0     258025.5
AMT_GOODS_PRICE              40500.0  450000.0    4050000.0
DAYS_BIRTH                  -25229.0  -15750.0      -7489.0
DAYS_EMPLOYED               -17912.0   -1213.0     365243.0
OWN_CAR_AGE                      0.0       0.0         91.0
FLAG_MOBIL                       0.0       1.0          1.0
FLAG_EMP_PHONE                   0.0       1.0          1.0
FLAG_WORK_PHONE                  0.0       0.0          1.0
FLAG_CONT_MOBILE                 0.0       1.0          1.0
FLAG_PHONE                       0.0       0.0          1.0
FLAG_EMAIL                       0.0       0.0          1.0
CNT_FAM_MEMBERS   

In [2]:
# Cell 2: Handle Anomalies and Convert Time Features

# 1. Fix the 365243 anomaly (Pensioners/Unemployed)
df_clean['DAYS_EMPLOYED'] = df_clean['DAYS_EMPLOYED'].replace(365243, 0)

# 2. Convert Negative Days to Positive Years (For better ML logic and Frontend UX)
df_clean['AGE_YEARS'] = (df_clean['DAYS_BIRTH'].abs() / 365).astype(int)
df_clean['YEARS_EMPLOYED'] = (df_clean['DAYS_EMPLOYED'].abs() / 365).round(2)

# Convert Phone Change to positive days 
df_clean['DAYS_LAST_PHONE_CHANGE'] = df_clean['DAYS_LAST_PHONE_CHANGE'].abs()

# 3. Drop the old confusing negative columns
df_clean = df_clean.drop(columns=['DAYS_BIRTH', 'DAYS_EMPLOYED'])

print("--- Anomaly & Time Conversion Complete ---")
print(df_clean[['AGE_YEARS', 'YEARS_EMPLOYED', 'DAYS_LAST_PHONE_CHANGE']].describe().loc[['min', 'max', 'mean']])

--- Anomaly & Time Conversion Complete ---
      AGE_YEARS  YEARS_EMPLOYED  DAYS_LAST_PHONE_CHANGE
min   20.000000        0.000000                0.000000
max   69.000000       49.070000             4292.000000
mean  43.435968        5.355752              962.858119


In [3]:
# Cell 3: Categorical Encoding (One-Hot Encoding)

print(f"Shape before encoding: {df_clean.shape}")

# 1. Identify categorical columns
# Note: TARGET is numeric (0/1), so it won't be affected
categorical_cols = df_clean.select_dtypes(include=['object', 'string']).columns

# 2. Apply One-Hot Encoding using pandas get_dummies
# drop_first=False keeps all categories explicit, which is safer for form integration later
df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, dummy_na=False)

# Convert boolean columns (True/False) to integers (1/0) if any were created
bool_cols = df_encoded.select_dtypes(include=['bool']).columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

print(f"Shape after encoding: {df_encoded.shape}")

# Let's peek at the new columns created
print("\nNew columns created (sample):")
print(df_encoded.columns.tolist()[-10:])

Shape before encoding: (307511, 40)
Shape after encoding: (307511, 157)

New columns created (sample):
['WALLSMATERIAL_MODE_Mixed', 'WALLSMATERIAL_MODE_Monolithic', 'WALLSMATERIAL_MODE_Others', 'WALLSMATERIAL_MODE_Panel', 'WALLSMATERIAL_MODE_Stone, brick', 'WALLSMATERIAL_MODE_Unknown', 'WALLSMATERIAL_MODE_Wooden', 'EMERGENCYSTATE_MODE_No', 'EMERGENCYSTATE_MODE_Unknown', 'EMERGENCYSTATE_MODE_Yes']


In [4]:
# Cell 4: Domain-Specific Feature Engineering

print(f"Shape before Feature Engineering: {df_encoded.shape}")

# 1. Debt-to-Income Ratio (How much of their income goes to the loan payment?)
df_encoded['ANNUITY_INCOME_PERCENT'] = df_encoded['AMT_ANNUITY'] / df_encoded['AMT_INCOME_TOTAL']

# 2. Credit-to-Income Ratio (How large is the loan compared to their salary?)
df_encoded['CREDIT_INCOME_PERCENT'] = df_encoded['AMT_CREDIT'] / df_encoded['AMT_INCOME_TOTAL']

# 3. Credit Term (Roughly how many years it will take to pay off the loan)
df_encoded['CREDIT_TERM'] = df_encoded['AMT_CREDIT'] / df_encoded['AMT_ANNUITY']

# 4. Employment Stability (What percentage of their life have they been working at their current job?)
# Safe to divide because AGE_YEARS min is 20 (no division by zero)
df_encoded['EMPLOYED_AGE_PERCENT'] = df_encoded['YEARS_EMPLOYED'] / df_encoded['AGE_YEARS']

# 5. Goods-to-Credit Ratio (Did they ask for a loan larger than the item they are buying?)
df_encoded['GOODS_CREDIT_PERCENT'] = df_encoded['AMT_GOODS_PRICE'] / df_encoded['AMT_CREDIT']

print(f"Shape after Feature Engineering: {df_encoded.shape}")

# Let's verify the new features
new_features = ['ANNUITY_INCOME_PERCENT', 'CREDIT_INCOME_PERCENT', 
                'CREDIT_TERM', 'EMPLOYED_AGE_PERCENT', 'GOODS_CREDIT_PERCENT']

print("\n--- Engineered Features Summary ---")
print(df_encoded[new_features].describe().loc[['min', 'mean', 'max', 'std']])

Shape before Feature Engineering: (307511, 157)
Shape after Feature Engineering: (307511, 162)

--- Engineered Features Summary ---
      ANNUITY_INCOME_PERCENT  CREDIT_INCOME_PERCENT  CREDIT_TERM  \
min                 0.000224               0.004808     6.324539   
mean                0.180929               3.957570    21.612366   
max                 1.875965              84.736842    59.560354   
std                 0.094573               2.689728     7.824164   

      EMPLOYED_AGE_PERCENT  GOODS_CREDIT_PERCENT  
min               0.000000              0.166667  
mean              0.130285              0.901702  
max               0.732388              6.666667  
std               0.136726              0.104845  


C:\Users\User\AppData\Local\Temp\ipykernel_30672\1424844649.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_encoded['ANNUITY_INCOME_PERCENT'] = df_encoded['AMT_ANNUITY'] / df_encoded['AMT_INCOME_TOTAL']
C:\Users\User\AppData\Local\Temp\ipykernel_30672\1424844649.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_encoded['CREDIT_INCOME_PERCENT'] = df_encoded['AMT_CREDIT'] / df_encoded['AMT_INCOME_TOTAL']
C:\Users\User\AppData\Local\Temp\ipykernel_30672\1424844649.py:12: PerformanceWarning: DataFrame is highly fragmente

In [5]:
# Cell 5: Save the fully preprocessed dataset for Model Training

# Save to CSV
df_encoded.to_csv('application_train_fully_preprocessed.csv', index=False)

# Also save the exact column names as a JSON list so the LightGBM model 
# knows EXACTLY what 161 features to expect during training and deployment.
import json
final_model_features = [col for col in df_encoded.columns if col != 'TARGET']

config_preprocessed = {
    "training_features": final_model_features,
    "target": "TARGET"
}

with open('model_training_features.json', 'w') as f:
    json.dump(config_preprocessed, f)

print("SUCCESS! Data saved as 'application_train_fully_preprocessed.csv'")
print("SUCCESS! Feature list saved as 'model_training_features.json'")

SUCCESS! Data saved as 'application_train_fully_preprocessed.csv'
SUCCESS! Feature list saved as 'model_training_features.json'
